In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

In [2]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
device = 'cuda'

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
# X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)

In [5]:
train_data = TensorDataset(X_train, y_train)
valid_data = TensorDataset(X_valid, y_valid)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)

In [6]:
class CancerClassifier(nn.Module):
    def __init__(self, n_features, n_units_l1, n_units_l2, dropout_rate):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, n_units_l1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(n_units_l1, n_units_l2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
        )
        self.output_layer = nn.Linear(n_units_l2, 1)
    
    def forward(self, X):
        deep = self.deep_stack(X)
        return self.output_layer(deep)

In [7]:
def train(model, optimizer, criterion, train_loader, valid_loader, n_epochs):
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        mean_train_loss = total_loss / len(train_loader)

        model.eval()
        total_valid_loss = 0
        total_correct = 0      # New: Keep track of right answers
        total_samples = 0      # New: Keep track of total patients tested
        
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                # 1. Get raw predictions and calculate loss
                raw_pred = model(X_batch)
                loss = criterion(raw_pred, y_batch)
                total_valid_loss += loss.item()
                
                # 2. ACCURACY MATH: Convert raw numbers to probabilities (0 to 1)
                probabilities = torch.sigmoid(raw_pred)
                
                # 3. Round to 0 or 1 (If > 0.5, guess 1. Else guess 0)
                guesses = probabilities.round()
                
                # 4. Count how many guesses match the actual y_batch
                correct_guesses = (guesses == y_batch).sum().item()
                
                total_correct += correct_guesses
                total_samples += len(y_batch) # Add the number of patients in this batch
                
        mean_valid_loss = total_valid_loss / len(valid_loader)
        
        # Calculate final accuracy percentage for the epoch
        accuracy = (total_correct / total_samples) * 100
        
        # Print the report!
        print(f"Epoch {epoch + 1:02d}/{n_epochs} | Train Loss: {mean_train_loss:.4f} | Valid Loss: {mean_valid_loss:.4f} | Accuracy: {accuracy:.2f}%")
    return accuracy

In [8]:
import optuna
def objective(trial):
    l1_neurons = trial.suggest_int("l1_neurons", 16, 64)
    l2_neurons = trial.suggest_int("l2_neurons", 4, 32)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)

    model = CancerClassifier(
        n_features=30,
        n_units_l1=l1_neurons,
        n_units_l2=l2_neurons,
        dropout_rate=dropout_rate
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    final_score = train(
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        train_loader=train_loader,
        valid_loader=valid_loader,
        n_epochs=20
    )
    return final_score


In [9]:
study = optuna.create_study(direction="maximize")
print("Starting Optuna search...")

study.optimize(objective, n_trials=20)

print("\n---WINNING SETUP---")
print(f"Best Accuracy: {study.best_value:.2f}%")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-04-21 23:06:10,553] A new study created in memory with name: no-name-a707bf78-3634-4cdc-a9a7-c1b7f56d290b


Starting Optuna search...
Epoch 01/20 | Train Loss: 0.7210 | Valid Loss: 0.6991 | Accuracy: 37.72%
Epoch 02/20 | Train Loss: 0.7071 | Valid Loss: 0.6778 | Accuracy: 37.72%
Epoch 03/20 | Train Loss: 0.6868 | Valid Loss: 0.6578 | Accuracy: 38.60%
Epoch 04/20 | Train Loss: 0.6688 | Valid Loss: 0.6394 | Accuracy: 41.23%
Epoch 05/20 | Train Loss: 0.6564 | Valid Loss: 0.6209 | Accuracy: 44.74%
Epoch 06/20 | Train Loss: 0.6408 | Valid Loss: 0.6027 | Accuracy: 51.75%
Epoch 07/20 | Train Loss: 0.6112 | Valid Loss: 0.5840 | Accuracy: 62.28%
Epoch 08/20 | Train Loss: 0.5988 | Valid Loss: 0.5647 | Accuracy: 77.19%
Epoch 09/20 | Train Loss: 0.5769 | Valid Loss: 0.5451 | Accuracy: 83.33%
Epoch 10/20 | Train Loss: 0.5733 | Valid Loss: 0.5250 | Accuracy: 89.47%
Epoch 11/20 | Train Loss: 0.5489 | Valid Loss: 0.5048 | Accuracy: 91.23%
Epoch 12/20 | Train Loss: 0.5277 | Valid Loss: 0.4844 | Accuracy: 92.11%
Epoch 13/20 | Train Loss: 0.5102 | Valid Loss: 0.4640 | Accuracy: 92.11%
Epoch 14/20 | Train Loss:

[I 2026-04-21 23:06:14,458] Trial 0 finished with value: 97.36842105263158 and parameters: {'l1_neurons': 59, 'l2_neurons': 15, 'dropout_rate': 0.32733359976709675, 'lr': 0.00015948544849592975}. Best is trial 0 with value: 97.36842105263158.


Epoch 16/20 | Train Loss: 0.4464 | Valid Loss: 0.4038 | Accuracy: 95.61%
Epoch 17/20 | Train Loss: 0.4215 | Valid Loss: 0.3841 | Accuracy: 95.61%
Epoch 18/20 | Train Loss: 0.4057 | Valid Loss: 0.3648 | Accuracy: 96.49%
Epoch 19/20 | Train Loss: 0.3884 | Valid Loss: 0.3462 | Accuracy: 96.49%
Epoch 20/20 | Train Loss: 0.3854 | Valid Loss: 0.3289 | Accuracy: 97.37%
Epoch 01/20 | Train Loss: 0.3940 | Valid Loss: 0.0356 | Accuracy: 98.25%
Epoch 02/20 | Train Loss: 0.2022 | Valid Loss: 0.0489 | Accuracy: 98.25%
Epoch 03/20 | Train Loss: 0.1332 | Valid Loss: 0.0739 | Accuracy: 95.61%
Epoch 04/20 | Train Loss: 0.1728 | Valid Loss: 0.0469 | Accuracy: 98.25%
Epoch 05/20 | Train Loss: 0.1076 | Valid Loss: 0.0534 | Accuracy: 99.12%
Epoch 06/20 | Train Loss: 0.1729 | Valid Loss: 0.0625 | Accuracy: 99.12%
Epoch 07/20 | Train Loss: 0.1069 | Valid Loss: 0.0803 | Accuracy: 98.25%
Epoch 08/20 | Train Loss: 0.1197 | Valid Loss: 0.0896 | Accuracy: 97.37%
Epoch 09/20 | Train Loss: 0.1652 | Valid Loss: 0.06

[I 2026-04-21 23:06:15,483] Trial 1 finished with value: 97.36842105263158 and parameters: {'l1_neurons': 45, 'l2_neurons': 29, 'dropout_rate': 0.4900788680791327, 'lr': 0.0523019890030671}. Best is trial 0 with value: 97.36842105263158.


...

Epoch 19/20 | Train Loss: 0.1221 | Valid Loss: 0.0684 | Accuracy: 98.25%
Epoch 20/20 | Train Loss: 0.1181 | Valid Loss: 0.0652 | Accuracy: 98.25%
Epoch 01/20 | Train Loss: 0.6176 | Valid Loss: 0.5351 | Accuracy: 90.35%
Epoch 02/20 | Train Loss: 0.4913 | Valid Loss: 0.3845 | Accuracy: 95.61%
Epoch 03/20 | Train Loss: 0.3574 | Valid Loss: 0.2588 | Accuracy: 95.61%
Epoch 04/20 | Train Loss: 0.2527 | Valid Loss: 0.1710 | Accuracy: 97.37%
Epoch 05/20 | Train Loss: 0.1997 | Valid Loss: 0.1204 | Accuracy: 97.37%
Epoch 06/20 | Train Loss: 0.1642 | Valid Loss: 0.0964 | Accuracy: 98.25%
Epoch 07/20 | Train Loss: 0.1485 | Valid Loss: 0.0840 | Accuracy: 98.25%
Epoch 08/20 | Train Loss: 0.1412 | Valid Loss: 0.0757 | Accuracy: 98.25%
Epoch 09/20 | Train Loss: 0.1441 | Valid Loss: 0.0746 | Accuracy: 98.25%
Epoch 10/20 | Train Loss: 0.1211 | Valid Loss: 0.0740 | Accuracy: 96.49%
Epoch 11/20 | Train Loss: 0.1114 | Valid Loss: 0.0688 | Accuracy: 98.25%
Epoch 12/20 | Train Loss: 0.1176 | Valid Loss: 0.06

[I 2026-04-21 23:06:32,842] Trial 18 finished with value: 97.36842105263158 and parameters: {'l1_neurons': 52, 'l2_neurons': 4, 'dropout_rate': 0.20169785776624422, 'lr': 0.001954367079745516}. Best is trial 14 with value: 99.12280701754386.


Epoch 17/20 | Train Loss: 0.1095 | Valid Loss: 0.0655 | Accuracy: 97.37%
Epoch 18/20 | Train Loss: 0.0974 | Valid Loss: 0.0652 | Accuracy: 97.37%
Epoch 19/20 | Train Loss: 0.1019 | Valid Loss: 0.0623 | Accuracy: 97.37%
Epoch 20/20 | Train Loss: 0.0965 | Valid Loss: 0.0613 | Accuracy: 97.37%
Epoch 01/20 | Train Loss: 0.6682 | Valid Loss: 0.6550 | Accuracy: 62.28%
Epoch 02/20 | Train Loss: 0.6375 | Valid Loss: 0.6164 | Accuracy: 77.19%
Epoch 03/20 | Train Loss: 0.5981 | Valid Loss: 0.5730 | Accuracy: 85.96%
Epoch 04/20 | Train Loss: 0.5640 | Valid Loss: 0.5179 | Accuracy: 92.98%
Epoch 05/20 | Train Loss: 0.5042 | Valid Loss: 0.4512 | Accuracy: 94.74%
Epoch 06/20 | Train Loss: 0.4566 | Valid Loss: 0.3772 | Accuracy: 95.61%
Epoch 07/20 | Train Loss: 0.3753 | Valid Loss: 0.3073 | Accuracy: 95.61%
Epoch 08/20 | Train Loss: 0.3370 | Valid Loss: 0.2487 | Accuracy: 96.49%
Epoch 09/20 | Train Loss: 0.2799 | Valid Loss: 0.2029 | Accuracy: 96.49%
Epoch 10/20 | Train Loss: 0.2404 | Valid Loss: 0.16

[I 2026-04-21 23:06:34,028] Trial 19 finished with value: 98.24561403508771 and parameters: {'l1_neurons': 40, 'l2_neurons': 8, 'dropout_rate': 0.3205937736433039, 'lr': 0.0007489939925738742}. Best is trial 14 with value: 99.12280701754386.


Epoch 17/20 | Train Loss: 0.1199 | Valid Loss: 0.0752 | Accuracy: 98.25%
Epoch 18/20 | Train Loss: 0.1190 | Valid Loss: 0.0710 | Accuracy: 98.25%
Epoch 19/20 | Train Loss: 0.1093 | Valid Loss: 0.0672 | Accuracy: 98.25%
Epoch 20/20 | Train Loss: 0.1016 | Valid Loss: 0.0645 | Accuracy: 98.25%

---WINNING SETUP---
Best Accuracy: 99.12%
  l1_neurons: 39
  l2_neurons: 6
  dropout_rate: 0.28465194140692274
  lr: 0.001647355845568894


In [10]:
best_params = study.best_params

print("Building final model with these parameters")
print(best_params)

final_model = CancerClassifier(
    n_features=30,
    n_units_l1=best_params["l1_neurons"],
    n_units_l2=best_params["l2_neurons"],
    dropout_rate=best_params["dropout_rate"]
).to(device)

final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])
criterion = nn.BCEWithLogitsLoss()

print("\n---TRAINING FINAL MODEL---")
final_accuracy = train(
    model=final_model,
    optimizer=final_optimizer,
    criterion=criterion,
    train_loader=train_loader,
    valid_loader=valid_loader,
    n_epochs=50
)

torch.save(final_model.state_dict(), "cancer_predictor.pth")
print("\nModel weigths saved to 'best_cancer_model.pth'")

Building final model with these parameters
{'l1_neurons': 39, 'l2_neurons': 6, 'dropout_rate': 0.28465194140692274, 'lr': 0.001647355845568894}

---TRAINING FINAL MODEL---
Epoch 01/50 | Train Loss: 0.6416 | Valid Loss: 0.5548 | Accuracy: 92.11%
Epoch 02/50 | Train Loss: 0.4912 | Valid Loss: 0.3864 | Accuracy: 96.49%
Epoch 03/50 | Train Loss: 0.3624 | Valid Loss: 0.2422 | Accuracy: 95.61%
Epoch 04/50 | Train Loss: 0.2587 | Valid Loss: 0.1544 | Accuracy: 97.37%
Epoch 05/50 | Train Loss: 0.2008 | Valid Loss: 0.1085 | Accuracy: 97.37%
Epoch 06/50 | Train Loss: 0.1519 | Valid Loss: 0.0848 | Accuracy: 97.37%
Epoch 07/50 | Train Loss: 0.1472 | Valid Loss: 0.0717 | Accuracy: 97.37%
Epoch 08/50 | Train Loss: 0.1349 | Valid Loss: 0.0637 | Accuracy: 97.37%
Epoch 09/50 | Train Loss: 0.1168 | Valid Loss: 0.0594 | Accuracy: 98.25%
Epoch 10/50 | Train Loss: 0.1026 | Valid Loss: 0.0576 | Accuracy: 98.25%
Epoch 11/50 | Train Loss: 0.0877 | Valid Loss: 0.0539 | Accuracy: 98.25%
Epoch 12/50 | Train Loss:

In [11]:
from torchsummary import summary

In [12]:
summary(final_model, (X.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 39]           1,209
              ReLU-2                   [-1, 39]               0
           Dropout-3                   [-1, 39]               0
            Linear-4                    [-1, 6]             240
              ReLU-5                    [-1, 6]               0
           Dropout-6                    [-1, 6]               0
            Linear-7                    [-1, 1]               7
Total params: 1,456
Trainable params: 1,456
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.01
Estimated Total Size (MB): 0.01
----------------------------------------------------------------
